In [79]:
# in case you are on google colab
%pip install qiskit pylatexenc

In [80]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, AncillaRegister, Parameter
from qiskit.quantum_info import Statevector, Operator
# in case you are on google colab
import pylatexenc
import math

from qiskit.circuit.library import QFTGate, UnitaryGate
# https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.QFTGate
# https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.UnitaryGate

# Import for adders - switching to DraperQFTAdder (handles carries internally)
from qiskit.circuit.library import DraperQFTAdder

import matplotlib.pyplot as plt

import numpy as np

In [ ]:
def grover_subset_sum(input_list, target_sum):
    max_bits = math.ceil(math.log2(max(input_list) + 1)) if max(input_list) > 0 else 1
    bits_target = math.ceil(math.log2(target_sum + 1)) if target_sum > 0 else 1
    bits_sum = math.ceil(math.log2(sum(input_list) + 1)) if sum(input_list) > 0 else 1
    rl = max(max_bits, bits_target, bits_sum)
    n = len(input_list)

    # FrrT (QFT with bit-reversal swaps)
    qr = QuantumRegister(rl, 'q')
    qc_h = QuantumCircuit(qr)
    for i in range(rl // 2):
        qc_h.swap(qr[i], qr[rl - 1 - i])
    for i in range(rl):
        qc_h.h(qr[i])
        for k in range(2, rl - i + 1):
            qc_h.cp(2 * math.pi / (2**k), qr[i + k - 1], qr[i])
    FrrT = qc_h.to_gate(label='FrrT')

    # U_adder
    qr_sum = QuantumRegister(rl, 'sum')
    qr_bool = QuantumRegister(n, 'bool')
    adder_circ = QuantumCircuit(qr_sum, qr_bool)
    adder_circ.append(FrrT, qr_sum)
    for k_idx, k_val in enumerate(input_list):
        for i in range(rl):
            adder_circ.cp(math.pi * k_val / (2**(rl - i - 1)), qr_bool[k_idx], qr_sum[i])
    adder_circ.append(FrrT.inverse(), qr_sum)
    U_adder = adder_circ.to_gate(label='U_adder')

    # U_oracle
    os_ = QuantumRegister(rl, 'sum')
    ob = QuantumRegister(n, 'bool')
    oracle_circ = QuantumCircuit(os_, ob)
    oracle_circ.append(U_adder, os_[:] + ob[:])
    bt = format(target_sum, f'0{rl}b')
    for i in range(rl):
        if bt[rl - 1 - i] == '0':
            oracle_circ.x(os_[i])
    oracle_circ.append(MCPhaseGate(math.pi, rl - 1), os_[:])
    for i in range(rl):
        if bt[rl - 1 - i] == '0':
            oracle_circ.x(os_[i])
    oracle_circ.append(U_adder.inverse(), os_[:] + ob[:])
    U_oracle = oracle_circ.to_gate(label='U_oracle')

    # G (Grover operator: oracle + diffuser)
    gs_ = QuantumRegister(rl, 'sum')
    gb = QuantumRegister(n, 'bool')
    g_circ = QuantumCircuit(gs_, gb)
    g_circ.append(U_oracle, gs_[:] + gb[:])
    g_circ.h(gb)
    g_circ.x(gb)
    g_circ.append(MCPhaseGate(math.pi, n - 1), gb[:])
    g_circ.x(gb)
    g_circ.h(gb)
    G = g_circ.to_gate(label='G')

    # Final circuit: superposition over bool, ancilla sum starts at |0>
    fsr = QuantumRegister(rl, 'sum')
    fbr = QuantumRegister(n, 'bool')
    qc_final = QuantumCircuit(fsr, fbr)
    qc_final.h(fbr)
    num_g_copies = round(math.pi / 4 * math.sqrt(2**n))
    for _ in range(num_g_copies):
        qc_final.append(G, fsr[:] + fbr[:])

    # Return all bool states with above-threshold probability
    psi = Statevector(qc_final)
    bool_probs = psi.probabilities(list(range(rl, rl + n)))
    threshold = 1 / (2 * 2**n)
    return [(bool_val, prob) for bool_val, prob in enumerate(bool_probs) if prob > threshold]

In [ ]:
results = grover_subset_sum([1, 3, 4], target_sum=4)
print(results)

In [86]:
new_sum_reg_for_oracle = QuantumRegister(register_length, name='sum')
new_bool_reg_for_oracle = QuantumRegister(len(input_list), name='bool')
qc_test_oracle = QuantumCircuit(new_sum_reg_for_oracle, new_bool_reg_for_oracle)

# Apply the Oracle
qc_test_oracle.append(U_oracle, new_sum_reg_for_oracle[:] + new_bool_reg_for_oracle[:])

bool_qubits = qc_test_oracle.qregs[1]
num_bool = len(bool_qubits)

# --- Diffuser for the boolean register ---
qc_test_oracle.h(bool_qubits)
qc_test_oracle.x(bool_qubits)

# Multi-controlled Z on the bool qubits
# Total qubits = num_bool, so controls = num_bool - 1
qc_test_oracle.append(MCPhaseGate(math.pi, num_bool - 1), bool_qubits[:])

qc_test_oracle.x(bool_qubits)
qc_test_oracle.h(bool_qubits)

G = qc_test_oracle.to_gate(label='G')

print("Grover operator G rebuilt:")
display(qc_test_oracle.draw('mpl'))

In [ ]:
cleaned = np.where(probabilities < 1e-20, 0, probabilities)
print("Probabilities (values < 1e-20 zeroed):")
print(cleaned)